# Tema 10 · Transformación de atributos: estandarización y componentes principales (PCA)

**Trabajo de investigación aplicada · Minería de Datos · grupal**

**Integrantes del grupo:** _escriba aquí los nombres_

## 📖 El caso

El **Hotel Costa Cobquecura** recibe turistas en la costa de Ñuble. Al dejar el hotel, cada huésped responde una encuesta de **12 preguntas** con nota de 1 a 7. La gerencia junta cientos de encuestas por temporada, pero doce promedios por mes son difíciles de leer y no sabe qué mirar. Quiere **resumir** la encuesta en pocos indicadores y saber cuáles se relacionan con que el huésped **vuelva**.

> **❓ La pregunta del cliente:** ¿Cuántas cosas distintas mide de verdad la encuesta, cómo se puede resumir en pocos indicadores y cuál de ellos conviene mirar para que los huéspedes vuelvan?

## 🗂️ Los datos: `t10_encuesta_hotel.csv` (480 encuestas con 12 preguntas de 1 a 7, noches, gasto y si el huésped volvería)

| Columna | Qué significa |
|---|---|
| `encuesta` | número de la encuesta |
| `recepcion` | nota a la atención en recepción (1 a 7) |
| `amabilidad` | nota a la amabilidad del personal |
| `rapidez_respuesta` | nota a la rapidez en responder solicitudes |
| `solucion_problemas` | nota a cómo se resolvieron los problemas |
| `habitacion` | nota a la habitación |
| `limpieza` | nota a la limpieza |
| `piscina_temperada` | nota a la piscina temperada |
| `calefaccion` | nota a la calefacción |
| `precio_noche` | nota al precio de la noche |
| `calidad_precio` | nota a la relación calidad-precio |
| `promociones` | nota a las promociones y descuentos |
| `precio_restaurante` | nota a los precios del restaurante |
| `noches` | noches que se quedó |
| `gasto_total_clp` | gasto total de la estadía, en pesos |
| `volveria` | **si el huésped dice que volvería (Sí/No)** ← lo que se quiere predecir en la sección 6 |

## Cómo trabajar este notebook

1. **Archivo → Guardar una copia en Drive.** Trabajen siempre en esa copia. Para trabajar en grupo, el dueño de la copia la **comparte** con el resto (botón **Compartir**).
2. **Todo el código viene escrito y explicado.** No hay que programar desde cero: lean la explicación 📘 que está antes de cada celda y ejecútenla (▶) **en orden**.
3. Las líneas **`# ✅ Debe mostrar:`** dicen qué debe aparecer. Si aparece otra cosa, hagan **Entorno de ejecución → Reiniciar y ejecutar todo**.
4. En las celdas **✏️ Experimento** cambien el valor que se indica, vuelvan a ejecutar y anoten lo que pasa en la tabla de abajo. **Al terminar, dejen el valor original.**
5. Respondan cada **✍️** con **sus números**. Esas respuestas son la materia prima de la **parte práctica del informe** y de la **exposición**.

| Ícono | Qué es |
|:-:|---|
| 📘 | Explicación de lo que hace la celda siguiente |
| ✅ | Lo que debe mostrar la celda |
| ✏️ | Experimento: ustedes cambian un valor |
| ✍️ | Pregunta para responder con sus números |
| 🧠 | De la información al conocimiento: convertir el resultado en una decisión para el cliente |

In [ ]:
# ▶ CELDA DE INICIO — no la modifique. Importa las librerías y carga los datos.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score

RUTA = "https://raw.githubusercontent.com/de-mag-ia/diguillin-datos/main/investigacion/"
datos = pd.read_csv(RUTA + "t10_encuesta_hotel.csv")

print(f"Datos cargados: {len(datos)} filas y {datos.shape[1]} columnas.")
# ✅ Debe mostrar: Datos cargados: 480 filas y 16 columnas.

## 1. Conocer los datos

📘 Cada fila es una **encuesta** de un huésped al dejar el hotel. Tiene **12 preguntas** con nota de 1 a 7, las
**noches** que se quedó, el **gasto total** en pesos y si **volvería** al hotel (Sí/No). La celda cuenta las encuestas.

In [ ]:
datos.head()

In [ ]:
preguntas = ["recepcion", "amabilidad", "rapidez_respuesta", "solucion_problemas",
             "habitacion", "limpieza", "piscina_temperada", "calefaccion",
             "precio_noche", "calidad_precio", "promociones", "precio_restaurante"]
print("Encuestas:", len(datos), "· Preguntas:", len(preguntas))
print("Volvería: Sí", (datos["volveria"] == "Sí").sum(), "· No", (datos["volveria"] == "No").sum())
# ✅ Debe mostrar: Encuestas: 480 · Preguntas: 12  /  Volvería: Sí 260 · No 220

## 2. ¿Las 12 preguntas dicen 12 cosas distintas?

📘 Se calcula la **correlación** entre cada par de preguntas (de −1 a 1; cerca de 1 = se mueven juntas) y se dibuja
como un mapa de colores: oscuro = correlación alta. Después se promedian las correlaciones entre preguntas del
**mismo grupo** de cuatro y entre preguntas de **grupos distintos**.

In [ ]:
correlacion = datos[preguntas].corr()

plt.figure(figsize=(7, 6))
plt.imshow(correlacion, cmap="Blues", vmin=0, vmax=1)
plt.colorbar(label="Correlación")
plt.xticks(range(12), preguntas, rotation=90)
plt.yticks(range(12), preguntas)
plt.title("Correlación entre las 12 preguntas")
plt.show()

grupos = [preguntas[0:4], preguntas[4:8], preguntas[8:12]]
dentro = [correlacion.loc[a, b] for g in grupos for a in g for b in g if a < b]
entre = [correlacion.loc[a, b] for i in range(3) for j in range(i + 1, 3) for a in grupos[i] for b in grupos[j]]
print(f"Correlación promedio dentro de cada grupo de 4: {np.mean(dentro):.2f}")
print(f"Correlación promedio entre grupos distintos:    {np.mean(entre):.2f}")
# ✅ Debe mostrar: Correlación promedio dentro de cada grupo de 4: 0.71  /  Correlación promedio entre grupos distintos:    0.02

### ✍️ ¿Qué muestra el mapa de colores? ¿Cuántas «cosas distintas» parece medir realmente la encuesta y cuáles son?

_Su respuesta:_

## 3. Primero, una trampa: columnas en escalas distintas

📘 El **análisis de componentes principales (PCA)** busca las direcciones en que los datos **varían más**. Se aplica
a todas las columnas numéricas tal como vienen: 12 notas de 1 a 7, las noches (1 a 7) y el **gasto en pesos**
(cientos de miles). La celda dice qué porcentaje de la variación total explica la primera componente y qué columna
pesa más en ella.

In [ ]:
numericas = datos.drop(columns=["encuesta", "volveria"])
pca_crudo = PCA().fit(numericas)
print(f"La componente 1 explica el {pca_crudo.explained_variance_ratio_[0]:.2%} de la variación")
print("Columna que más pesa en ella:", numericas.columns[np.abs(pca_crudo.components_[0]).argmax()])
print(f"Desviación estándar del gasto: {numericas['gasto_total_clp'].std():,.0f} pesos · de una pregunta: {datos[preguntas].std().mean():.1f} puntos")
# ✅ Debe mostrar: La componente 1 explica el 100.00% de la variación  /  Columna que más pesa en ella: gasto_total_clp  /  Desviación estándar del gasto: 137,489 pesos · de una pregunta: 1.3 puntos

### ✍️ ¿Por qué el PCA sin preparar los datos «descubre» solo el gasto? ¿Significa que el gasto es lo más importante del hotel?

_Su respuesta:_

## 4. Estandarizar y aplicar PCA a las 12 preguntas

📘 **Estandarizar** (`StandardScaler`) transforma cada columna para que tenga **promedio 0 y desviación estándar 1**:
a cada valor se le resta el promedio de la columna y se divide por su desviación estándar. Así todas las columnas
quedan en la misma escala. Luego se aplica el PCA a las 12 preguntas estandarizadas. Cada **componente** es una
columna **nueva**, construida como combinación de las 12 preguntas, y la celda dice cuánta variación resume cada una.

In [ ]:
estandarizadas = StandardScaler().fit_transform(datos[preguntas])
pca = PCA().fit(estandarizadas)

plt.figure(figsize=(7, 3.5))
plt.bar(range(1, 13), pca.explained_variance_ratio_, color="#1d5a7a")
plt.xticks(range(1, 13))
plt.xlabel("Componente")
plt.ylabel("Parte de la variación")
plt.title("¿Cuánta información resume cada componente?")
plt.show()

for i in range(4):
    print(f"Componente {i + 1}: {pca.explained_variance_ratio_[i]:.1%} de la variación")
print(f"Las 3 primeras juntas: {pca.explained_variance_ratio_[:3].sum():.1%}")
# ✅ Debe mostrar: Componente 1: 29.5% de la variación  /  Componente 2: 25.9% de la variación  /  Componente 3: 23.0% de la variación  /  Componente 4: 3.9% de la variación  /  Las 3 primeras juntas: 78.5%

## 5. ¿Qué significa cada componente?

📘 Cada componente tiene un **peso** para cada pregunta (`pca.components_`). Las preguntas con mayor peso (en valor
absoluto) dicen **qué mide** la componente. La celda muestra, para las 3 primeras, las 4 preguntas que más pesan.

In [ ]:
for i in range(3):
    pesos = pd.Series(np.abs(pca.components_[i]), index=preguntas)
    principales = [p for p in preguntas if p in pesos.nlargest(4).index]
    print(f"Componente {i + 1}:", ", ".join(principales))
# ✅ Debe mostrar: Componente 1: recepcion, amabilidad, rapidez_respuesta, solucion_problemas  /  Componente 2: habitacion, limpieza, piscina_temperada, calefaccion  /  Componente 3: precio_noche, calidad_precio, promociones, precio_restaurante

### ✍️ ¿Cuántas componentes conviene usar según el gráfico de barras y qué nombre le pondrían a cada una? ¿Cuánta información se pierde al pasar de 12 columnas a esas pocas?

_Su respuesta:_

## 6. ¿Sirven las componentes para predecir si el huésped volvería?

📘 Se entrena una **regresión logística** (un clasificador Sí/No) para predecir `volveria`, usando las **K primeras
componentes** en vez de las 12 preguntas. El acierto se mide con **validación cruzada de 5 pliegues** (el modelo se
prueba 5 veces con encuestas que no vio, y se promedia). Como referencia, primero se usa el modelo con las 12
preguntas completas.

In [ ]:
con_12 = make_pipeline(StandardScaler(), LogisticRegression())
print(f"Con las 12 preguntas: acierto {cross_val_score(con_12, datos[preguntas], datos['volveria'], cv=5).mean():.1%}")
# ✅ Debe mostrar: Con las 12 preguntas: acierto 81.2%

In [ ]:
# ✏️ EXPERIMENTO — cambie el valor indicado y vuelva a ejecutar
K = 3        # ✏️ pruebe con 1, 2, 3, 4, 6 y 12 (al terminar, deje 3)

con_pca = make_pipeline(StandardScaler(), PCA(n_components=K), LogisticRegression())
acierto = cross_val_score(con_pca, datos[preguntas], datos["volveria"], cv=5).mean()
print(f"Con {K} componentes: acierto {acierto:.1%}")

**✏️ Anoten aquí sus resultados:**

| Componentes (K) | Acierto (validación cruzada) |
|:-:|:-:|
| 1 | |
| 2 | |
| 3 | |
| 4 | |
| 6 | |
| 12 | |

### ✍️ ¿Con cuántas componentes se predice tan bien como con las 12 preguntas? La componente 3 (precio) es la que **menos** variación resume: ¿por qué, aun así, agregarla hace subir tanto el acierto?

_Su respuesta:_

### ✍️ 🧠 **De la información al conocimiento.** Escriban la recomendación para el hotel: ¿cómo conviene resumir y usar la encuesta, qué ganan con eso y qué deberían mirar primero para que los huéspedes vuelvan?

_Su respuesta:_

---
## ✔️ Para entregar

1. Revisen que **todas** las ✍️ tengan respuesta y que las tablas ✏️ estén completas.
2. **Entorno de ejecución → Reiniciar y ejecutar todo**: todo debe correr sin errores.
3. **Archivo → Descargar → Descargar .ipynb** y súbanlo al **Aula Virtual** como `Grupo_N_Tema10.ipynb`, junto con el informe en PDF.